---
## 0. Setup and imports

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    matthews_corrcoef, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# imblearn's Pipeline, not scikit-learn's: only this one applies a resampler,
# and only to the training part of each fold. sklearn's Pipeline ignores it.
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
N_FOLDS = 5

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

In [2]:
def summarise_folds(fold_scores: list[dict]) -> dict:
    """[{'pr_auc': 0.4, ...}, ...] -> {'pr_auc': {'mean': .., 'std': ..}, ...}"""
    if not fold_scores:
        raise ValueError("no fold scores to summarise")
    return {
        k: {"mean": float(np.mean([f[k] for f in fold_scores])),
            "std":  float(np.std([f[k] for f in fold_scores]))}
        for k in fold_scores[0]
    }


REPORT_METRICS = ("pr_auc", "roc_auc", "precision", "recall", "f1", "mcc", "accuracy")


def results_row(name: str, summary: dict, metrics=REPORT_METRICS) -> dict:
    """One row of the comparison table: 'mean ± std' strings, in report order.

    Metric order is deliberate. PR-AUC leads because at a ~3% positive rate it
    is the only headline that cannot be gamed by predicting the majority class;
    accuracy comes last for the same reason.
    """
    row = {"model": name}
    for m in metrics:
        if m in summary:
            row[m] = f"{summary[m]['mean']:.3f} ± {summary[m]['std']:.3f}"
    return row

---
## The evaluation harness

In [3]:
def evaluate(model, preprocessor, X, y, resampler=None, threshold=0.5) -> dict:
    """Stratified N_FOLDS-fold CV. Returns mean and std for every metric."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                          random_state=RANDOM_STATE)
    fold_scores, mats, curves = [], [], []

    for tr, va in skf.split(X, y):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr], y.iloc[va]

        # Rule 1: preprocessor fitted on the training part only.
        # Rule 2: resampler inside the pipeline, so it also sees training only.
        steps = [("pre", clone(preprocessor))]
        if resampler is not None:
            steps.append(("resample", clone(resampler)))
        steps.append(("clf", clone(model)))

        pipe = ImbPipeline(steps).fit(X_tr, y_tr)

        proba = pipe.predict_proba(X_va)[:, 1]
        pred = (proba >= threshold).astype(int)

        fold_scores.append({
            "pr_auc":    average_precision_score(y_va, proba),
            "roc_auc":   roc_auc_score(y_va, proba),
            "precision": precision_score(y_va, pred, zero_division=0),
            "recall":    recall_score(y_va, pred, zero_division=0),
            "f1":        f1_score(y_va, pred, zero_division=0),
            "mcc":       matthews_corrcoef(y_va, pred),
            "accuracy":  accuracy_score(y_va, pred),
        })
        mats.append(confusion_matrix(y_va, pred, labels=[0, 1]))
        p, r, _ = precision_recall_curve(y_va, proba)
        fpr, tpr, _ = roc_curve(y_va, proba)
        curves.append({"pr": (r, p), "roc": (fpr, tpr)})

    out = summarise_folds(fold_scores)
    out["confusion_matrix"] = np.sum(mats, axis=0)
    out["curves"] = curves
    return out


def show_confusion(summary: dict, label: str) -> pd.DataFrame:
    """Confusion matrix summed across folds, as a labelled frame."""
    cm = pd.DataFrame(summary["confusion_matrix"],
                      index=["actual: healthy", "actual: bankrupt"],
                      columns=["predicted: healthy", "predicted: bankrupt"])
    print(f"{label} — confusion matrix, summed over {N_FOLDS} folds")
    return cm

In [ ]:
def load_data() -> tuple[pd.DataFrame, pd.Series]:
    """Load the raw dataset. No scaling, no selection, no splitting."""
    df = pd.read_csv(DATA_PATH)

    # The published file has a leading space on nearly every column name.
    df.columns = [c.strip() for c in df.columns]

    y = df[TARGET].astype(int)
    X = df.drop(columns=[TARGET])

    # Fail loudly if this is not the file we think it is.
    assert X.shape == EXPECTED_SHAPE, f"expected {EXPECTED_SHAPE}, got {X.shape}"
    assert y.isin([0, 1]).all(), "target is not binary"
    return X, y


X, y = load_data()
print(f"features: {X.shape}   target: {y.shape}")

In [ ]:
print(f"rows: {X.shape[0]}   features: {X.shape[1]}")
print(f"missing values: {int(X.isna().sum().sum())}")
print(f"duplicate rows: {int(X.duplicated().sum())}")

counts = y.value_counts().sort_index()
pct = (y.value_counts(normalize=True).sort_index() * 100).round(2)
balance = pd.DataFrame({"count": counts, "percent": pct})
balance.index = ["not bankrupt (0)", "bankrupt (1)"]
print()
print(balance)

minority = float(pct.loc[1])
majority = float(pct.loc[0])
print(f"\nminority class: {minority:.2f}%")
print(f"a model that always predicts 0 would score {majority:.2f}% accuracy")

In [ ]:
# Distribution of the target, and a look at the scale spread across features.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["not bankrupt", "bankrupt"], counts.values,
            color=["#1F5B68", "#93382F"])
axes[0].set_ylabel("companies")
axes[0].set_title(f"Class balance — {minority:.2f}% positive")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f" {v:,}", ha="center", va="bottom")

ranges = (X.max() - X.min()).sort_values(ascending=False)
axes[1].hist(np.log10(ranges.replace(0, np.nan).dropna()), bins=30,
             color="#1F5B68")
axes[1].set_xlabel("log10(max - min) per feature")
axes[1].set_ylabel("features")
axes[1].set_title("Feature ranges differ by orders of magnitude")
plt.tight_layout()
plt.show()

print("widest-range features:")
print(ranges.head(5).to_string())
print("\nThis spread is why the linear model needs standardisation (lecture 2).")

In [ ]:
# Criterion 2 asks for the attributes described, not listed. Bucket the 95
# ratios by keyword so the report carries a table rather than 95 bullets.
GROUPS = {
    "Profitability": ["ROA", "profit", "margin", "income", "earning", "EPS",
                      "return"],
    "Leverage":      ["debt", "liabilit", "equity", "borrow", "interest-bearing",
                      "mortgage"],
    "Liquidity":     ["cash", "current ratio", "quick", "working capital"],
    "Efficiency":    ["turnover", "frequency", "rotation", "per share",
                      "expense rate"],
    "Growth":        ["growth rate", "increase"],
}


def bucket(col: str) -> str:
    low = col.lower()
    for name, keys in GROUPS.items():
        if any(k.lower() in low for k in keys):
            return name
    return "Other / structural"


attrs = pd.DataFrame({"attribute": X.columns})
attrs["group"] = attrs["attribute"].map(bucket)

summary = (attrs.groupby("group").size()
           .rename("n_features").sort_values(ascending=False)
           .to_frame())
summary["share"] = (summary["n_features"] / len(attrs) * 100).round(1)
print(summary)

print("\ntwo examples per group:")
for g, sub in attrs.groupby("group"):
    print(f"  {g}: {', '.join(sub['attribute'].head(2))}")